In [ ]:
import torch
import torch.nn as nn
from transformers import BeitImageProcessor, BeitForImageClassification
from torch.optim import AdamW

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize BEiT Components
# Note: BEiT has its own specific processor for normalization
feature_extractor_beit = BeitImageProcessor.from_pretrained('microsoft/beit-base-patch16-224')

model_beit = BeitForImageClassification.from_pretrained(
    'microsoft/beit-base-patch16-224',
    num_labels=4,
    ignore_mismatched_sizes=True
)
model_beit.to(device)

# 3. Optimizer and Loss
# BEiT often benefits from a slightly lower learning rate or higher weight decay
optimizer = AdamW(model_beit.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

print(f"✅ BEiT-Base loaded on {device}. Ready for training.")

In [ ]:
epochs = 25

for epoch in range(epochs):
    model_beit.train()
    train_loss, correct, total = 0, 0, 0

    for batch in train_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model_beit(inputs).logits
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Validation Phase
    model_beit.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for v_batch in val_dataloader:
            v_inputs, v_labels = v_batch
            v_inputs, v_labels = v_inputs.to(device), v_labels.to(device)
            v_outputs = model_beit(v_inputs).logits
            _, v_pred = v_outputs.max(1)
            val_total += v_labels.size(0)
            val_correct += v_pred.eq(v_labels).sum().item()

    print(f"BEiT Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_dataloader):.4f} | "
          f"Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from torch.utils.data import DataLoader

# Ensure model is in evaluation mode
model_beit.eval()

all_preds = []
all_labels = []
all_probs = []

print("Generating predictions for BEiT metrics...")

with torch.no_grad():
    for batch in val_dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass
        outputs = model_beit(inputs).logits
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays for sklearn
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

class_names = ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']

# --- Metric A: Classification Report ---
print("\nClassification Report for BEiT:\n")
report = classification_report(all_labels, all_preds, target_names=class_names)
print(report)

# --- Metric B: Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('BEiT Confusion Matrix')
plt.show()

# --- Metric C: Multi-class ROC Curve ---
plt.figure(figsize=(10, 8))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(all_labels == i, all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - BEiT')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()